# FightWatch — pilot hành động theo thời gian

Không train lớn nhiều giờ. Đối chứng checkpoint v1 với hai cấu hình: 16 frame/8fps và 16 frame/12fps. PKU chỉ có ba video nguồn: **0005-M train**, **0176-L/0360-R chẩn đoán**, giữ hai nguồn chẩn đoán cùng phía vì chưa xác minh độc lập diễn viên. Đây là feasibility nhỏ đã xem mẫu trước, không phải test cuối hoặc độ chính xác iPhone.

Dùng nhãn thời gian, không gán nhãn toàn video. Đấm/tát/đá **vào người** là dương; ôm/bắt tay/vỗ lưng/cử chỉ/đá đồ vật là âm. Push chưa rõ và confidence=0 để vùng không chấm. Mẫu lặp chỉ là lượt lấy mẫu cân bằng, không tăng số nguồn độc lập. Không tự cài candidate lên app.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, shutil
ROOT=Path('/content/drive/MyDrive/SENTINEL')
REPO=ROOT/'repo/anomaly-behavior-detector-system'
LOCAL=Path('/content/fightwatch-interaction-code')
BASE=ROOT/'artifacts/fight_large_v1'
SOURCE=ROOT/'datasets/fight (1)/pku_mmd_sample'
DATA=Path('/content/fightwatch-interaction-data')
CACHE=Path('/content/fightwatch-interaction-cache')
OUTPUT=ROOT/'artifacts/fight_interaction_pilot_v1'

## Pull code mới, giữ checkpoint/nguồn gốc
Chỉ bỏ thay đổi notebook tracked trên bản repo Drive; file code khác có thay đổi thì dừng. Baseline v1 và dataset không bị ghi đè.

In [ ]:
subprocess.run(['git','-C',str(REPO),'restore','--worktree','--','colab'],check=True)
changes=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
assert not changes.strip(), changes
subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
shutil.copytree(REPO,LOCAL,dirs_exist_ok=True,ignore=shutil.ignore_patterns('.git','.venv','logs','uploads','models','__pycache__','.pytest_cache'))
os.chdir(LOCAL)
from scripts.colab_process import run_logged
print('Runtime code:',REVISION)

## Chuẩn bị nếu chưa có kết quả
Ngân sách pilot 20 phút cho cache/train/chấm/xuất; stage dữ liệu/cài thư viện tách riêng. Backbone cache gắn SHA256 checkpoint khởi tạo, không dùng nhầm đặc trưng Kinetics với backbone đã fine-tune.

In [ ]:
if not (OUTPUT/'comparison.json').exists():
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
    import torch
    assert torch.cuda.is_available(), 'Cần GPU cho pilot'
    run_logged([sys.executable,'-m','scripts.fight_public_data','--archives',str(ROOT/'datasets/fight'),'--data','/content/fightwatch-public-data','--output',str(BASE)],OUTPUT/'data.log',1800,revision=REVISION)
    (DATA/'rgb').mkdir(parents=True,exist_ok=True)
    (DATA/'labels').mkdir(parents=True,exist_ok=True)
    for video in sorted((SOURCE/'rgb').glob('*.avi')):
        shutil.copyfile(video,DATA/'rgb'/video.name)
        labels=list((SOURCE/'labels').rglob(video.stem+'.txt'))
        assert len(labels)==1,(video.name,len(labels))
        shutil.copyfile(labels[0],DATA/'labels'/labels[0].name)
    shutil.copyfile(SOURCE/'Actions.xlsx',DATA/'Actions.xlsx')
    assert (BASE/'best.pt').exists()
    print('GPU:',torch.cuda.get_device_name(0))
else:
    print('Có kết quả: không tải dữ liệu hoặc giữ GPU để train lại.')

## Đối chứng có giới hạn
Checkpoint chọn bằng macro AP development RLVS/AIRTLab, một ngưỡng global chọn trên calibration cân bằng từng dataset. Không dùng test v1 để train/chọn/ngưỡng. Không có calibration PKU độc lập: không tự duyệt demo dù điểm tốt. Đánh giá thêm hai video liên tục theo nhãn thời gian và event gate giống runtime, cho phép trễ cuối sự kiện 1 giây, matching một-một IoU tối thiểu 0,1 trên khoảng mở rộng; đây không phải mAP chuẩn của paper.

In [ ]:
if not (OUTPUT/'comparison.json').exists():
    run_logged([sys.executable,'-m','scripts.fight_interaction_pilot','--baseline',str(BASE),'--pku',str(DATA),'--output',str(OUTPUT),'--cache',str(CACHE),'--budget-minutes','20'],OUTPUT/'training.log',1260,revision=REVISION)
else:
    print('Chỉ đọc kết quả; không tự chạy lại nếu completed/incomplete.')

In [ ]:
report=json.loads((OUTPUT/'comparison.json').read_text())
print('Status:',report['status'])
print('Baseline windows:',report.get('baseline_pku_windows'))
for name,variant in report['variants'].items():
    print('\nVariant:',name,'threshold:',variant['threshold'])
    print('Calibration:',variant['cohorts'])
    print('PKU diagnostic:',variant['pku_diagnostic'])
    for source in variant['continuous']:
        print(source['video'],source['metrics'])
print('Saved:',OUTPUT)
print('approved_for_demo=False; no physical phone tested.')

Checkpoint và report từng variant lưu trên Drive; không chọn model bằng cách đổi ngưỡng theo các nguồn chẩn đoán đã xem. Muốn nghiệm thu phải bổ sung nguồn/người diễn mới và thử iPhone thật sau khi kết nối. Nếu pilot không tốt hơn, giữ baseline; không tự train thêm nhiều giờ.

In [ ]:
run_logged([sys.executable,'-m','scripts.fight_interaction_report',str(OUTPUT)],OUTPUT/'report_render.log',120,revision=REVISION)
from IPython.display import Image, display
for image in sorted(OUTPUT.glob('*_timeline.png')):
    display(Image(filename=str(image)))
print('Màu đỏ: hành động cần phát hiện; xám: chưa chấm; cam: alarm runtime. Không phải kiểm thử iPhone thật.')